In [1]:
!pip install -q chromadb sentence-transformers rank_bm25


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
#mount drive
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


**Config + load the new master table**

In [2]:
# =========================
# PHASE 1 — CONFIG
# =========================

SNOMED_PATH = "/Users/vic/Desktop/NICE/CAM_C04_NICE_Project/dataset/processed/snomed/snomed_master_v3.csv"
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en"
CHROMA_COLLECTION_NAME = "snomed_master_v3_retrieval"
CHROMA_PERSIST_DIR = "/Users/vic/Desktop/NICE/CAM_C04_NICE_Project/chroma_db"
CHROMA_BATCH_SIZE = 5000
DEFAULT_TOP_K = 10

# =========================
# PHASE 1 — LOAD DATA
# =========================

import pandas as pd
import numpy as np

df = pd.read_csv(SNOMED_PATH)

print("Loaded rows:", df.shape[0])
display(df.head())

Loaded rows: 374846


,snomed_code,term,semantic_tag,parent_code,snomed_effective_date,in_qof,qof_cluster_id,qof_cluster_description,qof_inclusion_type,qof_included_under,in_opencodelists,opencodelist_clinical_areas,opencodelist_source_files,opencodelist_list_types,usage_count_nhs,log_usage_nhs,source_count
0,101009,Quilonia ethiopica (organism),organism,44577004,2002-01-31,False,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,0.0,0.0,0
1,102002,Hemoglobin Okaloosa (substance),substance,40830007,2002-01-31,False,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,0.0,0.0,0
2,103007,Squirrel fibroma virus (organism),organism,71511002,2002-01-31,False,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,0.0,0.0,0
3,104001,Excision of lesion of patella (procedure),procedure,125675003|68471001,2002-01-31,False,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,0.0,0.0,0
4,107008,Structure of fetal part of placenta (body stru...,body structure,119412005,2002-01-31,False,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,0.0,0.0,0


basic cleanup + type fixing













In [3]:
# =========================
# PHASE 2 — CLEANUP
# =========================

# keep only rows with usable code + term
df = df.dropna(subset=["snomed_code", "term"]).copy()

# standardise key columns
df["snomed_code"] = df["snomed_code"].astype(str).str.strip()
df["term"] = df["term"].astype(str).str.strip()
df["parent_code"] = df["parent_code"].astype(str).str.strip()

# fix booleans / numerics
df["in_qof"] = df["in_qof"].fillna(False).astype(bool)
df["in_opencodelists"] = df["in_opencodelists"].fillna(False).astype(bool)

df["usage_count_nhs"] = pd.to_numeric(df["usage_count_nhs"], errors="coerce").fillna(0)
df["log_usage_nhs"] = pd.to_numeric(df["log_usage_nhs"], errors="coerce").fillna(0)
df["source_count"] = pd.to_numeric(df["source_count"], errors="coerce").fillna(0).astype(int)

# dedupe just in case
df = df.drop_duplicates(subset=["snomed_code"]).copy()

print("Clean rows:", df.shape[0])
display(df.head())

Clean rows: 374827


,snomed_code,term,semantic_tag,parent_code,snomed_effective_date,in_qof,qof_cluster_id,qof_cluster_description,qof_inclusion_type,qof_included_under,in_opencodelists,opencodelist_clinical_areas,opencodelist_source_files,opencodelist_list_types,usage_count_nhs,log_usage_nhs,source_count
0,101009,Quilonia ethiopica (organism),organism,44577004,2002-01-31,False,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,0.0,0.0,0
1,102002,Hemoglobin Okaloosa (substance),substance,40830007,2002-01-31,False,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,0.0,0.0,0
2,103007,Squirrel fibroma virus (organism),organism,71511002,2002-01-31,False,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,0.0,0.0,0
3,104001,Excision of lesion of patella (procedure),procedure,125675003|68471001,2002-01-31,False,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,0.0,0.0,0
4,107008,Structure of fetal part of placenta (body stru...,body structure,119412005,2002-01-31,False,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,0.0,0.0,0


build better text for embedding

In [4]:
# =========================
# PHASE 3 — BUILD TEXT FOR EMBEDDING
# =========================

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

df["text_for_embedding"] = (
    df["term"].apply(safe_text) + " | " +
    df["semantic_tag"].apply(safe_text) + " | " +
    df["opencodelist_clinical_areas"].apply(safe_text) + " | " +
    df["qof_cluster_description"].apply(safe_text)
).str.strip()

print(df[["snomed_code", "term", "text_for_embedding"]].head())

  snomed_code                                               term  \
0      101009                      Quilonia ethiopica (organism)   
1      102002                    Hemoglobin Okaloosa (substance)   
2      103007                  Squirrel fibroma virus (organism)   
3      104001          Excision of lesion of patella (procedure)   
4      107008  Structure of fetal part of placenta (body stru...   

                                  text_for_embedding  
0      Quilonia ethiopica (organism) | organism |  |  
1   Hemoglobin Okaloosa (substance) | substance |  |  
2  Squirrel fibroma virus (organism) | organism |  |  
3  Excision of lesion of patella (procedure) | pr...  
4  Structure of fetal part of placenta (body stru...  


**Build embeddings**

 ⚠️ skip phase 4 and load saved embeddings
  (recomputing takes 10–15 minutes unnecessarily)

In [ ]:
# =========================
# PHASE 4 — BUILD EMBEDDINGS
# =========================

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = df["text_for_embedding"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=128
)

print("Embeddings shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2929 [00:00<?, ?it/s]

Embeddings shape: (374827, 384)


In [ ]:
# import numpy as np
# # save embeddings
# np.save("/content/drive/MyDrive/NICE/embeddings_v3.npy", embeddings)

In [ ]:
#print(len(df))
#print(embeddings.shape)

In [5]:
import numpy as np

embeddings = np.load("/Users/vic/Desktop/NICE/CAM_C04_NICE_Project/embeddings/embeddings_v3.npy")
print("Embeddings shape:", embeddings.shape)
print("DF shape:", df.shape)

Embeddings shape: (374827, 384)
DF shape: (374827, 18)


Quick quality check

In [6]:
test_idx = np.random.randint(0, len(df), 5)

for i in test_idx:
    print(df.iloc[i]["term"])

Abnormal results function studies of central nervous system (finding)
Transection (procedure)
Product containing precisely pantoprazole (as pantoprazole sodium) 40 milligram/1 vial powder for conventional release solution for injection (clinical drug)
Kyphoplasty of fracture of spine using fluoroscopic guidance (procedure)
Acute deep vein thrombosis of bilateral upper limbs following procedure (disorder)


In [7]:
assert embeddings.shape[0] == len(df), "Test Mismatch between embeddings and dataframe rows"

**build / load Chroma collection**

⚠️ Phase 5 takes > 20 min  

skip phase 5 and load saved chroma collection instead.

In [ ]:
# =========================
# PHASE 5 — BUILD / LOAD CHROMA
# =========================

import chromadb

client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

existing_collections = [c.name for c in client.list_collections()]

# delete old version if needed
# if CHROMA_COLLECTION_NAME in existing_collections:
 #   client.delete_collection(name=CHROMA_COLLECTION_NAME)
 #   print(f"Deleted old collection: {CHROMA_COLLECTION_NAME}")

print(f"Creating new collection: {CHROMA_COLLECTION_NAME}")
collection = client.create_collection(name=CHROMA_COLLECTION_NAME)

ids = df["snomed_code"].tolist()
documents = df["text_for_embedding"].tolist()
embedding_list = embeddings.tolist()

metadatas = [
    {
        "term": safe_text(row["term"]),
        "semantic_tag": safe_text(row["semantic_tag"]),
        "parent_code": safe_text(row["parent_code"]),
        "in_qof": str(bool(row["in_qof"])),
        "in_opencodelists": str(bool(row["in_opencodelists"])),
        "source_count": str(int(row["source_count"])),
        "usage_count_nhs": str(float(row["usage_count_nhs"]))
    }
    for _, row in df.iterrows()
]

for i in range(0, len(ids), CHROMA_BATCH_SIZE):
    collection.add(
        ids=ids[i:i + CHROMA_BATCH_SIZE],
        documents=documents[i:i + CHROMA_BATCH_SIZE],
        embeddings=embedding_list[i:i + CHROMA_BATCH_SIZE],
        metadatas=metadatas[i:i + CHROMA_BATCH_SIZE]
    )

print("New collection created.")
print("Chroma count:", collection.count())

Creating new collection: snomed_master_v3_retrieval
New collection created.
Chroma count: 374827


In [8]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

collection = client.get_collection(name=CHROMA_COLLECTION_NAME)

print("Loaded collection:", CHROMA_COLLECTION_NAME)
print("Chroma count:", collection.count())

Loaded collection: snomed_master_v3_retrieval
Chroma count: 374827


Test Chroma immediately

In [9]:
results = collection.query(
    query_texts=["type 2 diabetes"],
    n_results=5
)

results

/Users/vic/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:09<00:00, 9.08MiB/s]


{'ids': [['999480561000087100']],
 'embeddings': None,
 'documents': [['Nocardia niwae (organism) | organism |  |']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'in_qof': 'False',
    'source_count': '0',
    'usage_count_nhs': '0.0',
    'term': 'Nocardia niwae (organism)',
    'parent_code': '59674005',
    'semantic_tag': 'organism',
    'in_opencodelists': 'False'}]],
 'distances': [[1.7153255939483643]]}

**Build BM25 Retriever**

In [10]:
# =========================
# PHASE 6 — BUILD BM25
# =========================

from rank_bm25 import BM25Okapi
import numpy as np
import pandas as pd

tokenized_corpus = [
    str(text).lower().split()
    for text in df["text_for_embedding"].fillna("").tolist()
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 corpus size:", len(tokenized_corpus))

BM25 corpus size: 374827


**Semantic Retrieval Function**

In [12]:
# =========================
# PHASE 7 — SEMANTIC RETRIEVAL
# =========================

def semantic_retrieve(query, n=DEFAULT_TOP_K):
    """
    Return top-n semantic retrieval results from existing Chroma collection.
    Lower semantic_distance is better.
    """
    results = collection.query(
        query_texts=[query],
        n_results=n
    )

    output = []
    for i in range(len(results["ids"][0])):
        output.append({
            "snomed_code": results["ids"][0][i],
            "text_for_embedding": results["documents"][0][i],
            "semantic_distance": results["distances"][0][i]
        })

    sem_df = pd.DataFrame(output)

 # join back to full table for evidence
    sem_df = sem_df.merge(
        df,
        on="snomed_code",
        how="left"
    )

    return sem_df

In [13]:
# =========================
# PHASE 8 — BM25 RETRIEVAL
# =========================

def bm25_retrieve(query, n=DEFAULT_TOP_K):
    """
    Return top-n BM25 retrieval results.
    Higher bm25_score is better.
    """
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    top_idx = np.argsort(bm25_scores)[::-1][:n]

    results = df.iloc[top_idx].copy()
    results["bm25_score"] = bm25_scores[top_idx]

    return results.reset_index(drop=True)

In [15]:

# =========================
# PHASE 9 — HYBRID + EVIDENCE RERANKING
# =========================

def hybrid_retrieve(query, n_semantic=DEFAULT_TOP_K, n_bm25=DEFAULT_TOP_K):
    """
    Combine semantic + BM25 retrieval, then rerank using evidence.
    """
    sem_df = semantic_retrieve(query, n=n_semantic).copy()
    bm25_df = bm25_retrieve(query, n=n_bm25).copy()

    # semantic rank score
    sem_df = sem_df.drop_duplicates(subset=["snomed_code"]).reset_index(drop=True)
    sem_df["semantic_rank"] = range(1, len(sem_df) + 1)
    sem_df["semantic_rank_score"] = 1 / sem_df["semantic_rank"]

    # bm25 rank score
    bm25_df = bm25_df.drop_duplicates(subset=["snomed_code"]).reset_index(drop=True)
    bm25_df["bm25_rank"] = range(1, len(bm25_df) + 1)
    bm25_df["bm25_rank_score"] = 1 / bm25_df["bm25_rank"]

    # keep needed cols
    sem_cols = ["snomed_code", "semantic_distance", "semantic_rank_score"]
    bm25_cols = ["snomed_code", "bm25_score", "bm25_rank_score"]

    merged = pd.merge(
        sem_df[sem_cols],
        bm25_df[bm25_cols],
        on="snomed_code",
        how="outer"
    )

    # attach master evidence columns
    merged = merged.merge(df, on="snomed_code", how="left")

    # fill missing rank signals
    merged["semantic_rank_score"] = merged["semantic_rank_score"].fillna(0)
    merged["bm25_rank_score"] = merged["bm25_rank_score"].fillna(0)
    merged["semantic_distance"] = merged["semantic_distance"].fillna(np.nan)
    merged["bm25_score"] = merged["bm25_score"].fillna(0)

    # base hybrid score
    merged["hybrid_score"] = (
        merged["semantic_rank_score"] +
        merged["bm25_rank_score"]
    )

    # evidence-aware reranking
    merged["evidence_score"] = (
        merged["in_qof"].astype(int) * 3 +
        merged["in_opencodelists"].astype(int) * 2 +
        merged["log_usage_nhs"].fillna(0) * 0.5
    )

    # mild penalty for overly generic parent concepts
    merged["generic_penalty"] = merged["term"].str.lower().isin([
        "diabetes mellitus (disorder)",
        "asthma (disorder)",
        "hypertensive disorder (disorder)",
        "obesity (disorder)"
    ]).astype(int)

    merged["final_score"] = (
        merged["hybrid_score"] +
        merged["evidence_score"] -
        merged["generic_penalty"] * 1.0
    )

    merged = merged.sort_values("final_score", ascending=False).reset_index(drop=True)

    return merged


In [16]:
# =========================
# PHASE 10 — EXPLANATION LAYER
# =========================

def explain_result(row):
    reasons = []

    if row["in_qof"]:
        reasons.append("QOF supported")

    if row["in_opencodelists"]:
        areas = safe_text(row["opencodelist_clinical_areas"])
        if areas:
            reasons.append(f"OpenCodelists: {areas}")
        else:
            reasons.append("OpenCodelists supported")

    if row["usage_count_nhs"] > 0:
        reasons.append(f"NHS usage: {int(row['usage_count_nhs'])}")

    if safe_text(row["semantic_tag"]):
        reasons.append(f"Semantic tag: {row['semantic_tag']}")

    return "; ".join(reasons)

In [17]:
# =========================
# PHASE 11 — DECISION LABELS
# =========================

def label_code(row):
    if row["final_score"] >= 7:
        return "INCLUDE"
    elif row["final_score"] >= 4:
        return "REVIEW"
    else:
        return "EXCLUDE"

In [18]:

# =========================
# PHASE 12 — DEMO QUERIES
# =========================

queries = [
    "type 2 diabetes",
    "hypertension",
    "asthma",
    "obesity",
    "body mass index"
]

for q in queries:
    print(f"\nQUERY: {q}")

    results = hybrid_retrieve(q, n_semantic=10, n_bm25=10).copy()
    results["explanation"] = results.apply(explain_result, axis=1)
    results["decision"] = results.apply(label_code, axis=1)

    display(
        results[[
            "snomed_code",
            "term",
            "semantic_tag",
            "in_qof",
            "in_opencodelists",
            "usage_count_nhs",
            "final_score",
            "decision",
            "explanation"
        ]].head(10)
    )


QUERY: type 2 diabetes


,snomed_code,term,semantic_tag,in_qof,in_opencodelists,usage_count_nhs,final_score,decision,explanation
0,44054006,Diabetes mellitus type 2 (disorder),disorder,True,True,4666570.0,12.927968,INCLUDE,QOF supported; OpenCodelists: diabetes; NHS us...
1,443694000,Uncontrolled type 2 diabetes mellitus (disorder),disorder,True,True,37450.0,10.408251,INCLUDE,QOF supported; OpenCodelists: diabetes; NHS us...
2,140381000119104,Neuropathic ulcer of toe due to type 2 diabete...,disorder,True,True,250.0,8.762726,INCLUDE,QOF supported; OpenCodelists: diabetes; NHS us...
3,789569005,Neuropathic ulcer of heel due to type 2 diabet...,disorder,True,True,110.0,7.854765,INCLUDE,QOF supported; OpenCodelists: diabetes; NHS us...
4,140531000119105,Neuropathic ulcer of foot due to type 2 diabet...,disorder,True,True,60.0,7.388770,INCLUDE,QOF supported; OpenCodelists: diabetes; NHS us...
5,472969004,History of diabetes mellitus type 2 (situation),situation,False,True,6900.0,6.530822,REVIEW,OpenCodelists: diabetes; NHS usage: 6900; Sema...
6,445353002,Brittle type 2 diabetes mellitus (disorder),disorder,True,True,10.0,6.365614,REVIEW,QOF supported; OpenCodelists: diabetes; NHS us...
7,199230006,Pre-existing type 2 diabetes mellitus (disorder),disorder,False,True,310.0,5.069896,REVIEW,OpenCodelists: diabetes; NHS usage: 310; Seman...
8,237627000,Pregnancy and type 2 diabetes mellitus (disorder),disorder,False,True,120.0,4.522895,REVIEW,OpenCodelists: diabetes; NHS usage: 120; Seman...
9,359642000,Diabetes mellitus type 2 in nonobese (disorder),disorder,False,True,0.0,2.100000,EXCLUDE,OpenCodelists: diabetes; Semantic tag: disorder



QUERY: hypertension


,snomed_code,term,semantic_tag,in_qof,in_opencodelists,usage_count_nhs,final_score,decision,explanation
0,59621000,Essential hypertension (disorder),disorder,True,True,4231880.0,12.879079,INCLUDE,QOF supported; OpenCodelists: hypertension; NH...
1,162659009,Hypertension resolved (finding),finding,True,True,16550.0,10.057101,INCLUDE,QOF supported; OpenCodelists: hypertension; NH...
2,70272006,Malignant hypertension (disorder),disorder,True,True,5560.0,9.422878,INCLUDE,QOF supported; OpenCodelists: hypertension; NH...
3,31992008,Secondary hypertension (disorder),disorder,True,True,3340.0,9.182013,INCLUDE,QOF supported; OpenCodelists: hypertension; NH...
4,48146000,Diastolic hypertension (disorder),disorder,True,True,2340.0,9.045833,INCLUDE,QOF supported; OpenCodelists: hypertension; NH...
5,123799005,Renovascular hypertension (disorder),disorder,True,True,410.0,8.509297,INCLUDE,QOF supported; OpenCodelists: hypertension; NH...
6,762463000,Diastolic hypertension co-occurrent with systo...,disorder,True,True,60.0,8.055437,INCLUDE,QOF supported; OpenCodelists: hypertension; NH...
7,10725009,Benign hypertension (disorder),disorder,True,True,210.0,7.775929,INCLUDE,QOF supported; OpenCodelists: hypertension; NH...
8,28119000,Renal hypertension (disorder),disorder,True,True,190.0,7.768994,INCLUDE,QOF supported; OpenCodelists: hypertension; NH...
9,123800009,Goldblatt hypertension (disorder),disorder,True,True,0.0,5.333333,REVIEW,QOF supported; OpenCodelists: hypertension; Se...



QUERY: asthma


,snomed_code,term,semantic_tag,in_qof,in_opencodelists,usage_count_nhs,final_score,decision,explanation
0,195967001,Asthma (disorder),disorder,True,True,2879580.0,12.436578,INCLUDE,QOF supported; OpenCodelists: asthma; NHS usag...
1,370218001,Mild asthma (disorder),disorder,True,True,35720.0,10.491747,INCLUDE,QOF supported; OpenCodelists: asthma; NHS usag...
2,266361008,Non-allergic asthma (disorder),disorder,True,True,7220.0,9.775708,INCLUDE,QOF supported; OpenCodelists: asthma; NHS usag...
3,370219009,Moderate asthma (disorder),disorder,True,True,8260.0,9.709650,INCLUDE,QOF supported; OpenCodelists: asthma; NHS usag...
4,370221004,Severe asthma (disorder),disorder,True,True,5500.0,9.449200,INCLUDE,QOF supported; OpenCodelists: asthma; NHS usag...
5,370220003,Occasional asthma (disorder),disorder,True,True,3410.0,9.234047,INCLUDE,QOF supported; OpenCodelists: asthma; NHS usag...
6,57607007,Occupational asthma (disorder),disorder,True,True,1230.0,8.668902,INCLUDE,QOF supported; OpenCodelists: asthma; NHS usag...
7,41553006,Detergent asthma (disorder),disorder,True,True,10.0,6.698948,REVIEW,QOF supported; OpenCodelists: asthma; NHS usag...
8,233688007,Sulfite-induced asthma (disorder),disorder,True,True,10.0,6.323948,REVIEW,QOF supported; OpenCodelists: asthma; NHS usag...
9,18041002,Printers' asthma (disorder),disorder,True,True,0.0,5.100000,REVIEW,QOF supported; OpenCodelists: asthma; Semantic...



QUERY: obesity


,snomed_code,term,semantic_tag,in_qof,in_opencodelists,usage_count_nhs,final_score,decision,explanation
0,414916001,Obesity (disorder),disorder,False,False,265430.0,6.244555,REVIEW,NHS usage: 265430; Semantic tag: disorder
1,248311001,Central obesity (disorder),disorder,False,False,5290.0,4.486881,REVIEW,NHS usage: 5290; Semantic tag: disorder
2,444862003,Childhood obesity (disorder),disorder,False,False,1780.0,3.842465,EXCLUDE,NHS usage: 1780; Semantic tag: disorder
3,83911000119104,Severe obesity (disorder),disorder,False,False,40.0,2.190119,EXCLUDE,NHS usage: 40; Semantic tag: disorder
4,82793005,Hypothalamic obesity (disorder),disorder,False,False,20.0,1.633372,EXCLUDE,NHS usage: 20; Semantic tag: disorder
5,999480561000087100,Nocardia niwae (organism),organism,False,False,0.0,1.000000,EXCLUDE,Semantic tag: organism
6,414438005,Hyperplastic obesity (disorder),disorder,False,False,0.0,0.500000,EXCLUDE,Semantic tag: disorder
7,295509007,Hypertrophic obesity (disorder),disorder,False,False,0.0,0.250000,EXCLUDE,Semantic tag: disorder
8,248312008,Peripheral obesity (disorder),disorder,False,False,0.0,0.166667,EXCLUDE,Semantic tag: disorder
9,360566006,Buffalo obesity (disorder),disorder,False,False,0.0,0.142857,EXCLUDE,Semantic tag: disorder



QUERY: body mass index


,snomed_code,term,semantic_tag,in_qof,in_opencodelists,usage_count_nhs,final_score,decision,explanation
0,60621009,Body mass index (observable entity),observable entity,True,True,50833680.0,13.972035,INCLUDE,QOF supported; OpenCodelists: bmi; NHS usage: ...
1,162863004,Body mass index 25-29 - overweight (finding),finding,True,True,710010.0,12.069851,INCLUDE,QOF supported; OpenCodelists: bmi; NHS usage: ...
2,162864005,Body mass index 30+ - obesity (finding),finding,True,True,945560.0,12.004767,INCLUDE,QOF supported; OpenCodelists: bmi; NHS usage: ...
3,412768003,Body mass index 20-24 - normal (finding),finding,True,True,436070.0,11.692780,INCLUDE,QOF supported; OpenCodelists: bmi; NHS usage: ...
4,35425004,Normal body mass index (finding),finding,True,True,45960.0,11.367774,INCLUDE,QOF supported; OpenCodelists: bmi; NHS usage: ...
5,310252000,Body mass index less than 20 (finding),finding,True,True,99000.0,11.001443,INCLUDE,QOF supported; OpenCodelists: bmi; NHS usage: ...
6,301331008,Finding of body mass index (finding),finding,True,True,71930.0,10.702842,INCLUDE,QOF supported; OpenCodelists: bmi; NHS usage: ...
7,48499001,Increased body mass index (finding),finding,True,True,32450.0,10.336600,INCLUDE,QOF supported; OpenCodelists: bmi; NHS usage: ...
8,6497000,Decreased body mass index (finding),finding,True,True,5300.0,9.454492,INCLUDE,QOF supported; OpenCodelists: bmi; NHS usage: ...
9,427090001,Body mass index less than 16.5 (finding),finding,True,True,90.0,7.755430,INCLUDE,QOF supported; OpenCodelists: bmi; NHS usage: ...


Codes are ranked using a **hybrid retrieval and evidence-based** scoring approach.

First, candidates are retrieved using both **Semantic similarit**y and **BM25 keyword matching**.

These retrieval ranks are converted into reciprocal-rank style scores and combined.

The system then adds evidence-based weights for **QOF inclusion**, **OpenCodelists inclusion**, and **NHS usage frequency**, while applying a **small penalty** to **overly broad parent concepts**.

The final score therefore reflects both query relevance and external evidence strength.

## **Pod 1 — Retrieval & Ranking Baseline**

**Data sources used**


	•	SNOMED CT (concepts + hierarchy)
	•	QOF Business Rules (clinical inclusion signals)
	•	OpenCodelists (curated clinical code lists)
	•	NHS SNOMED usage data (real-world frequency)


**Features used in ranking**

	•	Semantic similarity (embedding-based retrieval)
	•	BM25 keyword relevance
	•	QOF inclusion (in_qof)
	•	OpenCodelists inclusion (in_opencodelists)
	•	NHS usage (usage_count_nhs, log-scaled)
	•	Basic concept filtering (generic parent penalty)

**Scoring logic**

	•	Combine semantic + BM25 rank scores
	•	Add evidence weights (QOF > OpenCodelists > usage)
	•	Apply small penalty to overly broad parent concepts
→ Final score = relevance + evidence − penalty


**Example outputs**

	•	Type 2 diabetes → core diagnosis + complications (neuropathy, CKD) ranked highly
	•	Hypertension → essential + variants (malignant, secondary) surfaced
	•	Asthma → severity levels (mild/moderate/severe) + monitoring codes included

Each result includes:

	•	INCLUDE / REVIEW / EXCLUDE label
	•	Evidence-based explanation (QOF, OpenCodelists, NHS usage)